# Advanced Python Enumerations: 22 Problems with Complete Solutions

**Topic:** Customizing and extending `enum.Enum`, based on the supplied lesson. **Level:** advanced; **format:** problem → implementation → explanation → executable checks. All exercises use only Python's standard library and run top-to-bottom in a fresh Jupyter kernel. Each problem uses distinct class names to avoid accidental redefinitions.

**Coverage:** member methods; identity and aliases; `__str__`/`__repr__`; safe ordering and `functools.total_ordering`; equality/hash invariants; truthiness; behavior-only base classes; member creation with `__new__` and `_value_`; metadata, lookups, `HTTPStatus`, serialization, validation, and an integrated state-machine capstone.

**How to study:** Read each challenge, hide its worked-solution cell, implement your own answer, then run the supplied checks. `assert` failures identify mismatches. Execute **Kernel → Restart & Run All** to verify everything. A green final cell means all checks completed successfully.

**Version:** Python 3.10+ (pattern matching is used in one problem). No pip installs, network access, or external files are needed. The notebook deliberately keeps outputs small.

In [1]:
import sys
from dataclasses import dataclass, replace
from enum import Enum, IntEnum, unique
from functools import total_ordering
from http import HTTPStatus
import json

assert sys.version_info >= (3, 10), "This workbook requires Python 3.10+."

CHECKED = []

def check(name, condition):
    """Record a verified invariant; raise immediately on failure."""
    assert condition, f"FAILED: {name}"
    CHECKED.append(name)

def raises(exc_type, function, *args, **kwargs):
    """Verify an expected failure without hiding unexpected exceptions."""
    try:
        function(*args, **kwargs)
    except exc_type:
        return True
    return False

print(f"Python {sys.version.split()[0]} — ready")

Python 3.13.7 — ready


## Part I — Core customization, identity, representation

In standard `Enum`, each named member is a singleton. Aliases share a singleton, whereas ordinary enum classes with equal *values* remain different types. Treat enums as semantic types, not merely bags of integers.

## Problem 01 — Bound member methods and safe display data

**Challenge.** Create a `PaintColor` enumeration with `RED`, `GREEN`, and `BLUE`. Implement `scaled(channel)` returning a dictionary keyed by the member; validate that the channel is an integer from 0 through 255 **without accepting `bool`**. Provide a readable `label` property. Verify bound-method behavior and erroneous inputs.

### Worked solution 01

In [2]:
class PaintColor(Enum):
    RED = 1
    GREEN = 2
    BLUE = 3

    @property
    def label(self):
        return self.name.title()

    def scaled(self, channel):
        if type(channel) is not int or not 0 <= channel <= 255:
            raise ValueError("channel must be an integer in [0, 255]")
        return {self: channel}

print(PaintColor.RED.scaled(128), PaintColor.BLUE.label)

{<PaintColor.RED: 1>: 128} Blue


**Why this works / key pitfalls.** The enum's member is the instance passed as `self`. Defining a method does not create a new enum member. Checking `type(x) is int` avoids the frequently overlooked fact that `bool` subclasses `int`.

In [3]:
check("P1 member method", PaintColor.RED.scaled(128) == {PaintColor.RED: 128})
check("P1 label", PaintColor.BLUE.label == "Blue")
check("P1 lower edge", PaintColor.GREEN.scaled(0)[PaintColor.GREEN] == 0)
check("P1 upper edge", PaintColor.GREEN.scaled(255)[PaintColor.GREEN] == 255)
check("P1 rejects bool", raises(ValueError, PaintColor.RED.scaled, True))
check("P1 rejects overflow", raises(ValueError, PaintColor.RED.scaled, 256))
check("P1 rejects negative", raises(ValueError, PaintColor.RED.scaled, -1))

## Problem 02 — Aliases, canonical iteration, and unique values

**Challenge.** Model an event enum in which `OPENED = 1`, `STARTED = 1`, and `CLOSED = 2`. Demonstrate alias identity and the distinction between iteration and `__members__`. Show how `@unique` catches accidental aliases, without making a deliberate demonstration halt the notebook.

### Worked solution 02

In [4]:
class AuditEvent(Enum):
    OPENED = 1
    STARTED = 1  # Intentional alias.
    CLOSED = 2

print("canonical:", [x.name for x in AuditEvent])
print("all defined names:", list(AuditEvent.__members__))
print("alias object identity:", AuditEvent.STARTED is AuditEvent.OPENED)

def build_duplicate_enum():
    @unique
    class StrictAuditEvent(Enum):
        OPENED = 1
        STARTED = 1
    return StrictAuditEvent

canonical: ['OPENED', 'CLOSED']
all defined names: ['OPENED', 'STARTED', 'CLOSED']
alias object identity: True


**Why this works / key pitfalls.** `Enum` turns duplicate values into aliases by default. Iteration excludes aliases, whereas `__members__` retains the mapping of every defined name. Apply `@unique` when each defined name must have a distinct value.

In [5]:
check("P2 same singleton", AuditEvent.STARTED is AuditEvent.OPENED)
check("P2 canonical iteration", len(list(AuditEvent)) == 2)
check("P2 names include aliases", list(AuditEvent.__members__) == ["OPENED", "STARTED", "CLOSED"])
check("P2 alias canonical name", AuditEvent.STARTED.name == "OPENED")
check("P2 unique guard", raises(ValueError, build_duplicate_enum))

## Problem 03 — Customize `str` and `repr` without corrupting identity

**Challenge.** Create a `WireState` enum that emits a stable lowercase wire string using `__str__`, while displaying diagnostic information using `__repr__`. Check that value lookup and identity work exactly as before. Contrast formatting against `.value` for serialization.

### Worked solution 03

In [6]:
class WireState(Enum):
    QUEUED = "queued"
    ACTIVE = "active"
    DONE = "done"

    def __str__(self):
        return self.value

    def __repr__(self):
        return f"WireState.{self.name}<{self.value!r}>"

print(str(WireState.QUEUED))
print(repr(WireState.QUEUED))
print(WireState("done"))

queued
WireState.QUEUED<'queued'>
done


**Why this works / key pitfalls.** `str` is presentation, and `repr` is diagnostics; neither should define the stored protocol representation. Serialize `.value` explicitly, because display methods are allowed to change independently of the wire format.

In [7]:
check("P3 str", str(WireState.ACTIVE) == "active")
check("P3 repr", repr(WireState.DONE) == "WireState.DONE<'done'>")
check("P3 lookup", WireState("done") is WireState.DONE)
check("P3 value stable", WireState.QUEUED.value == "queued")

## Problem 04 — Enum identity versus same underlying value

**Challenge.** Define separate enums `PrimaryCode` and `SecondaryCode`, each containing member `OK = 200`. Show why comparing raw values can be misleading. Implement `matches_code(member, value)` to check input types and compare without coercing arbitrary strings or booleans.

### Worked solution 04

In [8]:
class PrimaryCode(Enum):
    OK = 200

class SecondaryCode(Enum):
    OK = 200

def matches_code(member: PrimaryCode, value: int) -> bool:
    if not isinstance(member, PrimaryCode):
        raise TypeError("member must be a PrimaryCode")
    return type(value) is int and member.value == value

print(PrimaryCode.OK == SecondaryCode.OK)
print(PrimaryCode.OK.value == SecondaryCode.OK.value)

False
True


**Why this works / key pitfalls.** Two plain `Enum` member types are distinct even when their `.value` fields coincide. Explicitly validating a boundary argument keeps `True` from being treated as integer `1`, and prevents silent string coercion.

In [9]:
check("P4 distinct semantic type", PrimaryCode.OK != SecondaryCode.OK)
check("P4 matching values", PrimaryCode.OK.value == SecondaryCode.OK.value)
check("P4 strict matching", matches_code(PrimaryCode.OK, 200))
check("P4 string rejected", not matches_code(PrimaryCode.OK, "200"))
check("P4 wrong member rejected", raises(TypeError, matches_code, SecondaryCode.OK, 200))

## Part II — Ordering, equality, hashing, and truthiness

When overriding rich comparisons, return `NotImplemented` for unsupported operands; this gives Python's reflected operation a chance. Avoid comparing arbitrary enum subclasses together merely because both inherit from a common base class.

## Problem 05 — Total ordering with correct operand boundaries

**Challenge.** Implement `Severity` with numeric ranks 10, 20, 30. Make all six rich comparisons work by defining only the necessary ordering primitive plus `@total_ordering`. Never order against other enum classes or bare integers. Verify sorting, reflexive operations, and rejection of unsupported ordering.

### Worked solution 05

In [10]:
@total_ordering
class Severity(Enum):
    LOW = 10
    MEDIUM = 20
    HIGH = 30

    def __lt__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return self.value < other.value

ordered_severities = sorted([Severity.HIGH, Severity.LOW, Severity.MEDIUM])
print(ordered_severities)

[<Severity.LOW: 10>, <Severity.MEDIUM: 20>, <Severity.HIGH: 30>]


**Why this works / key pitfalls.** `Enum` already supplies identity-based equality. `total_ordering` derives `<=`, `>`, and `>=` from equality and `__lt__`. The strict type boundary avoids accidental cross-domain ordering, and `NotImplemented` is different from returning a false comparison result.

In [11]:
check("P5 sorted", ordered_severities == [Severity.LOW, Severity.MEDIUM, Severity.HIGH])
check("P5 <=", Severity.LOW <= Severity.MEDIUM)
check("P5 >= same", Severity.HIGH >= Severity.HIGH)
check("P5 >", Severity.HIGH > Severity.LOW)
check("P5 !=", Severity.LOW != Severity.HIGH)
check("P5 reject int", raises(TypeError, lambda: Severity.LOW < 10))
check("P5 reject other enum", raises(TypeError, lambda: Severity.LOW < PrimaryCode.OK))

## Problem 06 — Custom rank instead of misleading string order

**Challenge.** Create `Lifecycle` whose wire values are `"pending"`, `"processing"`, and `"complete"`. Define ordering using an explicit rank, independent of lexical order or definition positions. Make it total and ensure the wire values remain unchanged.

### Worked solution 06

In [12]:
@total_ordering
class Lifecycle(Enum):
    PENDING = ("pending", 0)
    PROCESSING = ("processing", 1)
    COMPLETE = ("complete", 2)

    @property
    def wire_value(self):
        return self.value[0]

    @property
    def rank(self):
        return self.value[1]

    def __lt__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return self.rank < other.rank

print([(member.wire_value, member.rank) for member in sorted(Lifecycle)])

[('pending', 0), ('processing', 1), ('complete', 2)]


**Why this works / key pitfalls.** Lexical ordering of strings does not encode business workflows; use an explicit rank. This version stores `(wire, rank)` as the enum value, so native `Lifecycle("pending")` will **not** work; later exercises solve that with `__new__` and `_value_`.

In [13]:
check("P6 semantic ranking", Lifecycle.PENDING < Lifecycle.PROCESSING < Lifecycle.COMPLETE)
check("P6 reverse", Lifecycle.COMPLETE > Lifecycle.PENDING)
check("P6 stable wire", Lifecycle.COMPLETE.wire_value == "complete")
check("P6 tuple lookup", Lifecycle(("pending", 0)) is Lifecycle.PENDING)
check("P6 string lookup not direct", raises(ValueError, Lifecycle, "pending"))

## Problem 07 — The equality/hash trap and a safe repair

**Challenge.** Demonstrate that a custom `__eq__` makes enum members unhashable unless a compatible `__hash__` is provided. Then implement a safe equality customization that only compares its own enum members, preserving hashability for dictionary keys and sets. Explain why comparing equal to raw integers complicates the contract.

### Worked solution 07

In [14]:
class UnhashableSignal(Enum):
    OFF = 0
    ON = 1

    def __eq__(self, other):
        if type(self) is type(other):
            return self is other
        return NotImplemented

class HashableSignal(Enum):
    OFF = 0
    ON = 1

    def __eq__(self, other):
        if type(self) is type(other):
            return self is other
        return NotImplemented

    __hash__ = Enum.__hash__  # Compatible with the identity-based equality above.

signal_counts = {HashableSignal.OFF: 4, HashableSignal.ON: 9}
print(signal_counts[HashableSignal.ON])

9


**Why this works / key pitfalls.** Python sets `__hash__ = None` when `__eq__` is overridden without defining `__hash__`. Here equality remains identity-based, so reusing `Enum.__hash__` is compatible. If you instead make `EnumMember == 1`, its hash must match `hash(1)` and its equality must remain symmetric/transitive across other numeric values: usually an unnecessary risk. Keep `.value` comparisons at I/O boundaries.

In [15]:
check("P7 override unhashable", raises(TypeError, hash, UnhashableSignal.ON))
check("P7 repaired hash", isinstance(hash(HashableSignal.ON), int))
check("P7 key lookup", signal_counts[HashableSignal.OFF] == 4)
check("P7 member identity", HashableSignal.ON == HashableSignal.ON)
check("P7 no raw equality", HashableSignal.ON != 1)
check("P7 distinct hashes do not imply distinct", len({HashableSignal.OFF, HashableSignal.ON}) == 2)

## Problem 08 — Truthiness is not an enum value test

**Challenge.** Construct `RequestReadiness` where `READY = 1` and `BLOCKED = 0`. First prove that a plain `Enum` member holding zero remains true. Then implement `__bool__` with intentional semantics. Demonstrate explicit equality in safety-critical calling code.

### Worked solution 08

In [16]:
class DefaultReadiness(Enum):
    READY = 1
    BLOCKED = 0

class RequestReadiness(Enum):
    READY = 1
    BLOCKED = 0

    def __bool__(self):
        return self is RequestReadiness.READY

def may_launch(state: RequestReadiness) -> bool:
    if not isinstance(state, RequestReadiness):
        raise TypeError("state must be RequestReadiness")
    return state is RequestReadiness.READY

print("default:", bool(DefaultReadiness.BLOCKED))
print("custom:", bool(RequestReadiness.BLOCKED))

default: True
custom: False


**Why this works / key pitfalls.** For standard `Enum`, truthiness does not derive from `.value`. A custom boolean method can be convenient, but an explicit identity test is often clearer when permissions or state transitions are involved. Do not infer that `IntEnum` has the same truthiness behavior: it inherits integer behavior.

In [17]:
check("P8 default zero truthy", bool(DefaultReadiness.BLOCKED))
check("P8 custom zero false", not bool(RequestReadiness.BLOCKED))
check("P8 custom ready true", bool(RequestReadiness.READY))
check("P8 explicit allowed", may_launch(RequestReadiness.READY))
check("P8 explicit blocked", not may_launch(RequestReadiness.BLOCKED))
check("P8 reject wrong type", raises(TypeError, may_launch, DefaultReadiness.READY))

## Problem 09 — Subclassing constraints and reusable behavior

**Challenge.** Show that an enum class containing members cannot be extended by subclassing, even just to add another member. Build a member-free `NamedEnumBase` with a reusable `describe()` method, then derive two concrete enums with distinct members.

### Worked solution 09

In [18]:
class FixedColor(Enum):
    RED = 1
    BLUE = 2

def bad_subclass():
    class ExtendedFixedColor(FixedColor):
        GREEN = 3
    return ExtendedFixedColor

class NamedEnumBase(Enum):
    def describe(self):
        return f"{type(self).__name__}:{self.name}={self.value!r}"

class Direction(NamedEnumBase):
    NORTH = "N"
    SOUTH = "S"

class SignalMode(NamedEnumBase):
    AUTO = 1
    MANUAL = 2

print(Direction.NORTH.describe(), SignalMode.AUTO.describe())

Direction:NORTH='N' SignalMode:AUTO=1


**Why this works / key pitfalls.** An enum with members is closed to enum subclassing: preserving the finite member set is part of its contract. A base class **without** members may carry methods, properties, and `__new__` behavior reused by concrete enumerations.

In [19]:
check("P9 cannot extend members", raises(TypeError, bad_subclass))
check("P9 inherited method A", Direction.NORTH.describe() == "Direction:NORTH='N'")
check("P9 inherited method B", SignalMode.AUTO.describe() == "SignalMode:AUTO=1")
check("P9 classes distinct", Direction is not SignalMode)

## Problem 10 — Reusable ordering base that rejects foreign enum classes

**Challenge.** Extract ordering into a member-free `StrictOrderedEnum` and derive both `BuildStage` and `SizeClass`. Ensure same-class comparisons work but comparing a stage with a size raises `TypeError`, even though both are instances of the base class.

### Worked solution 10

In [20]:
@total_ordering
class StrictOrderedEnum(Enum):
    def __lt__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return self.value < other.value

class BuildStage(StrictOrderedEnum):
    SOURCE = 1
    COMPILE = 2
    PACKAGE = 3

class SizeClass(StrictOrderedEnum):
    SMALL = 1
    MEDIUM = 2
    LARGE = 3

print(BuildStage.SOURCE < BuildStage.PACKAGE)
print(SizeClass.LARGE >= SizeClass.MEDIUM)

True
True


**Why this works / key pitfalls.** A common but subtle bug is `isinstance(other, StrictOrderedEnum)`: it allows comparison across unrelated subclasses. Requiring `type(self) is type(other)` explicitly prevents comparing ranks belonging to unrelated domains.

In [21]:
check("P10 derived ordering", BuildStage.SOURCE < BuildStage.COMPILE)
check("P10 derived <=", SizeClass.SMALL <= SizeClass.LARGE)
check("P10 no cross domain", raises(TypeError, lambda: BuildStage.SOURCE < SizeClass.SMALL))
check("P10 no raw numeric order", raises(TypeError, lambda: SizeClass.SMALL < 2))

## Part III — Multiple fields, `__new__`, and robust lookups

An enum declaration such as `OK = (0, "Success")` uses the **whole tuple** as `.value` by default. A custom `__new__` can instead create the member, set its primary `_value_`, and attach additional attributes before the enum class is finalized. Do not call `super().__new__()` to create an enum member in this pattern; for plain `Enum`, use `object.__new__(cls)`.

## Problem 11 — Diagnose tuple-value lookup failure

**Challenge.** Create a simple `TupleStatus` with `(code, phrase)` tuple values, expose `.code` and `.phrase` as properties, then show why `TupleStatus(0)` fails and why tuple lookup succeeds. Do not silently claim the numeric lookup works.

### Worked solution 11

In [22]:
class TupleStatus(Enum):
    OK = (0, "Operation succeeded")
    FAILED = (1, "Operation failed")

    @property
    def code(self):
        return self.value[0]

    @property
    def phrase(self):
        return self.value[1]

print(TupleStatus.OK.value)
print(TupleStatus.OK.code, TupleStatus.OK.phrase)
print("code alone is not the stored value")

(0, 'Operation succeeded')
0 Operation succeeded
code alone is not the stored value


**Why this works / key pitfalls.** Properties split fields for display but cannot change the enum's stored lookup key. Construction by raw value requires the **entire** tuple: `TupleStatus((0, 'Operation succeeded'))`.

In [23]:
check("P11 tuple is value", TupleStatus.OK.value == (0, "Operation succeeded"))
check("P11 metadata", TupleStatus.FAILED.phrase == "Operation failed")
check("P11 code lookup fails", raises(ValueError, TupleStatus, 0))
check("P11 tuple lookup succeeds", TupleStatus((0, "Operation succeeded")) is TupleStatus.OK)

## Problem 12 — Implement `__new__` and `_value_` correctly

**Challenge.** Repair numeric lookup using `AppResult(Enum)` with members `(0, "Success", False)`, `(5, "Bad input", False)`, and `(9, "Try again", True)`. Expose `.phrase`, `.retryable`, and a useful `as_dict()` method. Verify that `.value` is the integer rather than the tuple.

### Worked solution 12

In [24]:
@unique
class AppResult(Enum):
    OK = (0, "Success", False)
    BAD_INPUT = (5, "Bad input", False)
    RETRY = (9, "Try again", True)

    def __new__(cls, code, phrase, retryable):
        if type(code) is not int or type(phrase) is not str or type(retryable) is not bool:
            raise TypeError("expected (int code, str phrase, bool retryable)")
        obj = object.__new__(cls)
        obj._value_ = code
        obj.phrase = phrase
        obj.retryable = retryable
        return obj

    def as_dict(self):
        return {"code": self.value, "message": self.phrase, "retryable": self.retryable}

print(AppResult(9).as_dict())

{'code': 9, 'message': 'Try again', 'retryable': True}


**Why this works / key pitfalls.** `Enum` passes tuple declaration fields as positional arguments to `__new__`. `_value_` sets the actual lookup key; the other instance attributes hold metadata. The enum's built-in value constructor now resolves integers naturally. `@unique` guards against accidental duplicate codes.

In [25]:
check("P12 int key", AppResult.OK.value == 0)
check("P12 numeric constructor", AppResult(9) is AppResult.RETRY)
check("P12 full metadata", AppResult.RETRY.as_dict() == {"code": 9, "message": "Try again", "retryable": True})
check("P12 invalid code", raises(ValueError, AppResult, 99))
check("P12 three members", len(list(AppResult)) == 3)

## Problem 13 — A reusable two-field enum with constructor validation

**Challenge.** Factor `(code, phrase)` construction into member-free `CodePhraseEnum`. Subclass it twice as `StorageCode` and `PaymentCode`; ensure they have separate semantic types, scalar `.value`s, and readable descriptions. Add code/phrase validation and `@unique` to the concrete classes.

### Worked solution 13

In [26]:
class CodePhraseEnum(Enum):
    def __new__(cls, code, phrase):
        if type(code) is not int or type(phrase) is not str or not phrase.strip():
            raise ValueError("expected integer code and nonempty phrase")
        obj = object.__new__(cls)
        obj._value_ = code
        obj.phrase = phrase
        return obj

    def describe(self):
        return f"{self.value}: {self.phrase}"

@unique
class StorageCode(CodePhraseEnum):
    STORED = (100, "Stored")
    MISSING = (404, "Not in storage")

@unique
class PaymentCode(CodePhraseEnum):
    PAID = (100, "Payment received")
    DECLINED = (402, "Payment required")

print(StorageCode(404).describe())

404: Not in storage


**Why this works / key pitfalls.** No values appear on the shared base, so enum subclassing is permitted. The integer `100` has meanings in two distinct enum domains; plain `Enum` keeps those members unequal even with equal raw `.value`s.

In [27]:
check("P13 storage lookup", StorageCode(100) is StorageCode.STORED)
check("P13 payment lookup", PaymentCode(100) is PaymentCode.PAID)
check("P13 cross-domain distinct", StorageCode.STORED != PaymentCode.PAID)
check("P13 phrase", PaymentCode.DECLINED.phrase == "Payment required")
check("P13 invalid code", raises(ValueError, PaymentCode, 555))

## Problem 14 — Detect duplicate scalar codes at class definition time

**Challenge.** Build a function that *attempts* to declare an enum with two differently described members using code `7` from `CodePhraseEnum`, then capture the expected `@unique` failure. Explain why storing extra metadata does not automatically make the codes unique.

### Worked solution 14

In [28]:
def build_duplicate_codes():
    @unique
    class DuplicateCode(CodePhraseEnum):
        FIRST = (7, "First meaning")
        SECOND = (7, "Different meaning")
    return DuplicateCode

print("duplicates rejected:", raises(ValueError, build_duplicate_codes))

duplicates rejected: True


**Why this works / key pitfalls.** Aliases are computed from the finalized `_value_`, not all constructor arguments. Both entries assign `_value_ = 7`; the second becomes an alias and `@unique` rejects the declaration. Distinct textual phrases do not make the enum keys distinct.

In [29]:
check("P14 duplicate code caught", raises(ValueError, build_duplicate_codes))

## Problem 15 — Normalization via `_missing_` without accepting garbage

**Challenge.** Define `Region` with `US`, `EU`, `APAC` and stable lower-case string values. Support case-insensitive string lookup with surrounding whitespace using `_missing_`. Reject numbers, partial matches, and unknown strings. Explain why `_missing_` is a value-lookup hook, not a general input-validation substitute.

### Worked solution 15

In [30]:
@unique
class Region(Enum):
    US = "us"
    EU = "eu"
    APAC = "apac"

    @classmethod
    def _missing_(cls, value):
        if not isinstance(value, str):
            return None
        normalized = value.strip().casefold()
        for member in cls:
            if member.value == normalized:
                return member
        return None

print(Region("  EU  "), Region("ApAc"))

Region.EU Region.APAC


**Why this works / key pitfalls.** `_missing_` runs after ordinary value lookup fails. It may return an existing member (or `None` to allow `ValueError`). Limit accepted transformations to those intentionally part of the API. A loop is clear for small enums and avoids fragile dynamic mutation of Enum internals.

In [31]:
check("P15 casefold", Region("Eu") is Region.EU)
check("P15 whitespace", Region(" apac  ") is Region.APAC)
check("P15 unknown rejects", raises(ValueError, Region, "EUP"))
check("P15 number rejects", raises(ValueError, Region, 1))
check("P15 name lookup still works", Region["US"] is Region.US)

## Problem 16 — Strict external-code parsing and bool/int collisions

**Challenge.** Demonstrate the Python rule `True == 1`: an integer-valued `Enum` may accept boolean input through value lookup. Write `parse_storage_code(raw)` that accepts only an actual `int`, raises `TypeError` on bool/float/str, and raises `ValueError` on unknown integer codes.

### Worked solution 16

In [32]:
class BooleanCollision(Enum):
    ZERO = 0
    ONE = 1

def parse_storage_code(raw):
    if type(raw) is not int:
        raise TypeError("storage code must be an integer, not bool or text")
    return StorageCode(raw)

print("Python numeric collision:", BooleanCollision(True) is BooleanCollision.ONE)
print("strict code parsing:", parse_storage_code(404))

Python numeric collision: True
strict code parsing: StorageCode.MISSING


**Why this works / key pitfalls.** Plain value lookup uses Python value equivalence: `True` hashes and compares like `1`. This can matter for JSON, where booleans and integers are different conceptual types. Validate the original input **before** constructing your enum if the protocol requires an exact integer.

In [33]:
check("P16 collision demonstrated", BooleanCollision(True) is BooleanCollision.ONE)
check("P16 valid strict parse", parse_storage_code(404) is StorageCode.MISSING)
check("P16 bool rejected", raises(TypeError, parse_storage_code, True))
check("P16 string rejected", raises(TypeError, parse_storage_code, "404"))
check("P16 float rejected", raises(TypeError, parse_storage_code, 404.0))
check("P16 unknown rejected", raises(ValueError, parse_storage_code, 999))

## Part IV — Interoperability and application design

`HTTPStatus` supplies a real standard-library example of an enum with numeric values and associated textual metadata. At boundaries, validate the exact external type and serialize `.value` explicitly. `IntEnum` and string-mixin enums intentionally trade some enum isolation for interoperability with primitive types.

## Problem 17 — Analyze `http.HTTPStatus` like a library author

**Challenge.** Inspect code, name, phrase, numeric reconstruction, and error behavior for `HTTPStatus.NOT_FOUND`. Implement `http_summary(raw)` to accept actual integers only and return a minimal JSON-safe dict. Validate an unknown code rather than inventing a status.

### Worked solution 17

In [34]:
def http_summary(raw):
    if type(raw) is not int:
        raise TypeError("HTTP code must be an integer")
    status = HTTPStatus(raw)
    return {"code": status.value, "name": status.name, "phrase": status.phrase}

print(http_summary(404))
print("http enum name:", HTTPStatus.NOT_FOUND.name)

{'code': 404, 'name': 'NOT_FOUND', 'phrase': 'Not Found'}
http enum name: NOT_FOUND


**Why this works / key pitfalls.** `HTTPStatus` (an `IntEnum`) has scalar numeric codes and a `.phrase` attribute, precisely the shape motivating the lesson's custom enum. The exact set of supported statuses can change with Python versions; a code like `999` is deliberately unknown here.

In [35]:
check("P17 numeric member", HTTPStatus(404) is HTTPStatus.NOT_FOUND)
check("P17 expected phrase", http_summary(404)["phrase"] == "Not Found")
check("P17 dict stable", http_summary(200)["code"] == 200)
check("P17 bool rejects", raises(TypeError, http_summary, True))
check("P17 unknown rejects", raises(ValueError, http_summary, 999))

## Problem 18 — Choose `Enum`, `IntEnum`, and a str-mixin intentionally

**Challenge.** Implement one opaque enum, one integer-compatible enum, and one string-compatible enum. Illustrate differences in equality to primitives, instance checks, and arithmetic. Define a JSON encoder that serializes an ordinary opaque enum through `.value`.

### Worked solution 18

In [36]:
class OpaqueId(Enum):
    ITEM = 1

class CompatibleId(IntEnum):
    ITEM = 1

class ContentType(str, Enum):
    JSON = "application/json"
    TEXT = "text/plain"

payload = {"id": OpaqueId.ITEM.value, "type": ContentType.JSON.value}
encoded_payload = json.dumps(payload)
print("opaque comparison:", OpaqueId.ITEM == 1)
print("integer-compatible:", CompatibleId.ITEM == 1)
print("encoded:", encoded_payload)

opaque comparison: False
integer-compatible: True
encoded: {"id": 1, "type": "application/json"}


**Why this works / key pitfalls.** `IntEnum` compares to integers and arithmetic generally produces an `int`, not a member. `class ContentType(str, Enum)` is a portable string-mixin pattern. Prefer plain `Enum` when accidental primitive equality would be harmful. For transport, serialize `.value` (and validate types on read).

In [37]:
check("P18 opaque inequality", OpaqueId.ITEM != 1)
check("P18 int equality", CompatibleId.ITEM == 1)
check("P18 arithmetic drops type", type(CompatibleId.ITEM + 1) is int)
check("P18 string compatibility", ContentType.JSON == "application/json")
check("P18 string instance", isinstance(ContentType.JSON, str))
check("P18 JSON roundtrip", json.loads(encoded_payload) == payload)

## Problem 19 — Robust JSON round-tripping for enum records

**Challenge.** Build `encode_app_result` and `decode_app_result` functions around `AppResult`. The encoder must produce only JSON-compatible data. The decoder must reject invalid JSON structures, missing/extra fields, boolean codes, and unknown codes, while treating the enum metadata as authoritative rather than trusting client-supplied phrases.

### Worked solution 19

In [38]:
def encode_app_result(result):
    if not isinstance(result, AppResult):
        raise TypeError("expected AppResult")
    return json.dumps({"code": result.value})

def decode_app_result(raw_json):
    obj = json.loads(raw_json)
    if not isinstance(obj, dict) or set(obj) != {"code"}:
        raise ValueError("expected object containing exactly 'code'")
    if type(obj["code"]) is not int:
        raise TypeError("code must be an integer")
    return AppResult(obj["code"])

encoded_result = encode_app_result(AppResult.RETRY)
decoded_result = decode_app_result(encoded_result)
print(encoded_result, decoded_result.phrase)

{"code": 9} Try again


**Why this works / key pitfalls.** Transport contains only the stable code; the trusted enum defines the phrase and retry policy. Schema validation guards against untrusted metadata, extra keys, booleans masquerading as integers, and unsupported codes. Validation of JSON syntax is delegated to `json.loads`, which raises `JSONDecodeError` for malformed text.

In [39]:
check("P19 roundtrip", decoded_result is AppResult.RETRY)
check("P19 stable JSON", json.loads(encoded_result) == {"code": 9})
check("P19 extra rejected", raises(ValueError, decode_app_result, '{"code": 0, "phrase": "forged"}'))
check("P19 missing rejected", raises(ValueError, decode_app_result, '{}'))
check("P19 bool rejected", raises(TypeError, decode_app_result, '{"code": true}'))
check("P19 unknown rejected", raises(ValueError, decode_app_result, '{"code": 12345}'))
check("P19 malformed rejected", raises(json.JSONDecodeError, decode_app_result, '{bad'))

## Problem 20 — Keep runtime state out of enum singletons

**Challenge.** Create an immutable `Worker` dataclass with `name` and `status`, and a `WorkerStatus` enum carrying a `can_accept()` method. Make `assign(worker)` return a **new** worker marked BUSY when READY; reject assignment when BUSY. Show why storing a per-request mutable counter directly on a singleton enum member would be shared globally.

### Worked solution 20

In [40]:
class WorkerStatus(Enum):
    READY = "ready"
    BUSY = "busy"

    def can_accept(self):
        return self is WorkerStatus.READY

@dataclass(frozen=True)
class Worker:
    name: str
    status: WorkerStatus

    def __post_init__(self):
        if not isinstance(self.name, str) or not self.name.strip():
            raise ValueError("worker name must be nonempty")
        if not isinstance(self.status, WorkerStatus):
            raise TypeError("invalid worker status")

def assign(worker):
    if not isinstance(worker, Worker):
        raise TypeError("expected Worker")
    if not worker.status.can_accept():
        raise RuntimeError("worker is already busy")
    return replace(worker, status=WorkerStatus.BUSY)

idle = Worker("alpha", WorkerStatus.READY)
busy = assign(idle)
print(idle, "=>", busy)

Worker(name='alpha', status=<WorkerStatus.READY: 'ready'>) => Worker(name='alpha', status=<WorkerStatus.BUSY: 'busy'>)


**Why this works / key pitfalls.** Every reference to a given enum member points to the same singleton. Per-worker changing data therefore belongs in ordinary instances, not attributes mutated on the enum member. Frozen dataclasses make state transitions explicit and avoid unexpected modifications of the previous worker object.

In [41]:
check("P20 new status", busy.status is WorkerStatus.BUSY)
check("P20 original preserved", idle.status is WorkerStatus.READY)
check("P20 reject double assignment", raises(RuntimeError, assign, busy))
check("P20 immutable", raises(Exception, setattr, idle, "status", WorkerStatus.BUSY))
check("P20 invalid status rejected", raises(TypeError, Worker, "bad", "ready"))

## Problem 21 — Use pattern matching with exhaustive validation

**Challenge.** Write `describe_outcome(outcome)` using Python 3.10 structural pattern matching to handle each `AppResult` explicitly and reject unsupported objects. Demonstrate that adding a new member later would require updating matching logic when relevant, and that the wildcard is not a silent success path.

### Worked solution 21

In [42]:
def describe_outcome(outcome):
    if not isinstance(outcome, AppResult):
        raise TypeError("expected AppResult")
    match outcome:
        case AppResult.OK:
            return "completed"
        case AppResult.BAD_INPUT:
            return "check the request"
        case AppResult.RETRY:
            return "retry later"
        case _:
            raise ValueError(f"unhandled result: {outcome!r}")

print({member.name: describe_outcome(member) for member in AppResult})

{'OK': 'completed', 'BAD_INPUT': 'check the request', 'RETRY': 'retry later'}


**Why this works / key pitfalls.** Value patterns such as `case AppResult.OK:` match existing enum members. Validate the incoming type first, and handle the wildcard explicitly. Python does not statically enforce exhaustive matching for `Enum`; tests can check that every current member is handled.

In [43]:
check("P21 all cases handled", all(isinstance(describe_outcome(m), str) for m in AppResult))
check("P21 retry branch", describe_outcome(AppResult.RETRY) == "retry later")
check("P21 reject nonmember", raises(TypeError, describe_outcome, 9))

## Part V — Integrated capstone

The final challenge combines behavior-only inheritance, a custom enum constructor, rank and metadata, explicit transition rules, immutable per-job state, secure boundary validation, and JSON round-tripping. Implementing all the pieces together tests whether they form a coherent API rather than a collection of isolated tricks.

## Problem 22 — Capstone: typed job lifecycle with validated transitions

**Challenge.** Build a compact job-processing API meeting **all** of these requirements:

1. A member-free base attaches a scalar string wire value, an integer rank, and a human-readable phrase via `__new__`.
2. A concrete `JobPhase` enum has `NEW`, `QUEUED`, `RUNNING`, `DONE`, and `FAILED`, with unique wire values and comparison restricted to its own type.
3. Define explicit legal transitions (`NEW→QUEUED`, `QUEUED→RUNNING`, `RUNNING→DONE/FAILED`) independently of ordering; terminal states have no exits.
4. Keep `job_id` and phase in a frozen data record. Validate strictly at construction and transitions, and never mutate enum member instances.
5. Encode a job as a two-field JSON object and decode it with strict schema/type checks. Unsupported values or transitions fail loudly.
6. Demonstrate a successful job, a failed job, unknown payloads, same-state transitions, cross-class ordering errors, and preservation of the original record.

### Worked solution 22

In [44]:
@total_ordering
class JobPhaseBase(Enum):
    """Behavior-only base: no enum members are declared here."""

    def __new__(cls, wire, rank, phrase):
        if type(wire) is not str or not wire:
            raise ValueError("wire value must be nonempty text")
        if type(rank) is not int or rank < 0:
            raise ValueError("rank must be a nonnegative integer")
        if type(phrase) is not str or not phrase.strip():
            raise ValueError("phrase must be nonempty text")
        obj = object.__new__(cls)
        obj._value_ = wire
        obj.rank = rank
        obj.phrase = phrase
        return obj

    def __lt__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return (self.rank, self.name) < (other.rank, other.name)

    def describe(self):
        return f"{self.value}: {self.phrase}"

@unique
class JobPhase(JobPhaseBase):
    NEW = ("new", 0, "Job created")
    QUEUED = ("queued", 1, "Awaiting a worker")
    RUNNING = ("running", 2, "Processing")
    DONE = ("done", 3, "Completed successfully")
    FAILED = ("failed", 3, "Terminated unsuccessfully")

# Explicit business rules: comparable ranks are NOT transitions.
ALLOWED_TRANSITIONS = {
    JobPhase.NEW: frozenset({JobPhase.QUEUED}),
    JobPhase.QUEUED: frozenset({JobPhase.RUNNING}),
    JobPhase.RUNNING: frozenset({JobPhase.DONE, JobPhase.FAILED}),
    JobPhase.DONE: frozenset(),
    JobPhase.FAILED: frozenset(),
}

@dataclass(frozen=True)
class Job:
    job_id: str
    phase: JobPhase

    def __post_init__(self):
        if type(self.job_id) is not str or not self.job_id.strip():
            raise ValueError("job_id must be nonempty text")
        if not isinstance(self.phase, JobPhase):
            raise TypeError("phase must be a JobPhase member")

    def transition(self, next_phase):
        if not isinstance(next_phase, JobPhase):
            raise TypeError("next_phase must be JobPhase")
        if next_phase not in ALLOWED_TRANSITIONS[self.phase]:
            raise ValueError(f"illegal transition: {self.phase.name} -> {next_phase.name}")
        return replace(self, phase=next_phase)

def encode_job(job):
    if not isinstance(job, Job):
        raise TypeError("expected Job")
    return json.dumps({"job_id": job.job_id, "phase": job.phase.value}, sort_keys=True)

def decode_job(raw_json):
    payload = json.loads(raw_json)
    if type(payload) is not dict or set(payload) != {"job_id", "phase"}:
        raise ValueError("expected exactly job_id and phase")
    if type(payload["job_id"]) is not str or not payload["job_id"].strip():
        raise ValueError("job_id must be nonempty text")
    if type(payload["phase"]) is not str:
        raise TypeError("phase must be a string")
    return Job(payload["job_id"], JobPhase(payload["phase"]))

initial = Job("job-001", JobPhase.NEW)
queued = initial.transition(JobPhase.QUEUED)
running = queued.transition(JobPhase.RUNNING)
succeeded = running.transition(JobPhase.DONE)
failed = running.transition(JobPhase.FAILED)
serialized_job = encode_job(succeeded)
print("successful:", serialized_job)
print("failure:", failed.phase.describe())

successful: {"job_id": "job-001", "phase": "done"}
failure: failed: Terminated unsuccessfully


**Why this works / key pitfalls.** The enum carries shared *definitions* (rank, phrase, wire identifier). The `Job` dataclass carries mutable-in-time *application state* but uses immutable snapshots. Rank is a display/order concept, never permission to transition; `DONE` and `FAILED` deliberately share rank yet have distinct unique wire keys. Strict schema and type checks keep transport safe; storing only wire values prevents client data from overriding trusted phrases. `@total_ordering` preserves enum identity equality, while a `(rank, name)` tie-breaker creates a consistent total ordering even when two members share a business rank. The follow-up illustrates equivalent sorting via an explicit key.

In [45]:
check("P22 base has no members", list(JobPhaseBase) == [])
check("P22 scalar value", JobPhase("running") is JobPhase.RUNNING)
check("P22 shared rank", JobPhase.DONE.rank == JobPhase.FAILED.rank == 3)
check("P22 distinct tied members ordered", JobPhase.DONE < JobPhase.FAILED)
check("P22 total ordering coherent", JobPhase.DONE <= JobPhase.FAILED and not (JobPhase.FAILED <= JobPhase.DONE))
check("P22 unique values", len(list(JobPhase)) == 5)
check("P22 sorted rank", JobPhase.NEW < JobPhase.RUNNING)
check("P22 success path", succeeded.phase is JobPhase.DONE)
check("P22 failure path", failed.phase is JobPhase.FAILED)
check("P22 original unchanged", initial.phase is JobPhase.NEW)
check("P22 roundtrip", decode_job(serialized_job) == succeeded)
check("P22 unknown phase", raises(ValueError, decode_job, '{"job_id":"x","phase":"missing"}'))
check("P22 extra key", raises(ValueError, decode_job, '{"job_id":"x","phase":"new","admin":true}'))
check("P22 wrong type", raises(TypeError, decode_job, '{"job_id":"x","phase":1}'))
check("P22 invalid job id", raises(ValueError, decode_job, '{"job_id":"","phase":"new"}'))
check("P22 cannot skip phase", raises(ValueError, initial.transition, JobPhase.RUNNING))
check("P22 cannot re-enter terminal", raises(ValueError, succeeded.transition, JobPhase.QUEUED))
check("P22 cannot repeat phase", raises(ValueError, queued.transition, JobPhase.QUEUED))
check("P22 no wrong enum", raises(TypeError, initial.transition, WorkerStatus.BUSY))
check("P22 no cross ordering", raises(TypeError, lambda: JobPhase.NEW < BuildStage.SOURCE))
check("P22 phrase stable", JobPhase.FAILED.phrase == "Terminated unsuccessfully")

### Capstone refinement — an explicit sorting key

`@total_ordering` assumes `__eq__` agrees with strict ordering. The capstone solves this with the `(rank, name)` tie-breaker in `__lt__`; an explicit sorting key makes the same ordering choice at call sites, without changing identity or wire values. If shared rank is all that matters to the business process, compare `.rank` explicitly rather than treating tied phases as identical.

In [46]:
def sorted_job_phases():
    """Deterministic total order, even when business ranks tie."""
    return sorted(JobPhase, key=lambda member: (member.rank, member.name))

ordered_jobs = sorted_job_phases()
check("P22 refinement deterministic order", len(ordered_jobs) == 5)
check("P22 refinement strict tiebreak", ordered_jobs.index(JobPhase.DONE) < ordered_jobs.index(JobPhase.FAILED))
check("P22 refinement still distinct", JobPhase.DONE is not JobPhase.FAILED)
print([member.name for member in ordered_jobs])

['NEW', 'QUEUED', 'RUNNING', 'DONE', 'FAILED']


## Extra worked examples — quick comparisons of implementation choices

These short demos are complementary to the challenges, not replacements for their edge-case tests.

In [47]:
# Example A: name lookup and value lookup are different operations.
print("by name:", AppResult["RETRY"])
print("by value:", AppResult(9))
check("example A", AppResult["RETRY"] is AppResult(9))

# Example B: external text normalization is explicitly scoped to Region.
print("normalized region:", Region("  uS "))
check("example B", Region("  uS ") is Region.US)

# Example C: metadata does not replace the underlying scalar key.
print("stored scalar:", StorageCode.MISSING.value)
print("human text:", StorageCode.MISSING.phrase)
check("example C", type(StorageCode.MISSING.value) is int)

# Example D: even hashable enum keys remain specific to their class.
counters = {PaintColor.RED: 2, WireState.DONE: 3}
check("example D", counters[PaintColor.RED] == 2 and counters[WireState.DONE] == 3)

# Example E: successful JSON is type-checked on the way back.
wire = encode_app_result(AppResult.OK)
check("example E", decode_app_result(wire) is AppResult.OK)
print("all five examples succeeded")

by name: AppResult.RETRY
by value: AppResult.RETRY
normalized region: Region.US
stored scalar: 404
human text: Not in storage
all five examples succeeded


## Review checklist and takeaways

- **Identity:** distinguish enum members and their primitive `.value`; avoid unnecessary cross-type `__eq__` overrides.
- **Contracts:** `__eq__` and `__hash__` must agree. Use `NotImplemented` for unsupported rich-comparison operands.
- **Ordering:** prefer explicit semantic ranks; use `@total_ordering` only when your equality and ordering definitions are consistent. Keep ordering within a concrete enum type.
- **Boolean behavior:** plain `Enum(0)` members are truthy by default. Implement `__bool__` only deliberately; prefer explicit state tests for permissions.
- **Inheritance:** derive concrete enumerations from behavior-only enum bases, never from an enum that already defines members.
- **Member creation:** when tuple declarations should be looked up by a scalar, create the member in `__new__`, assign `_value_`, and attach metadata. Use `@unique` for distinct codes.
- **Boundaries:** validate JSON input types (especially `bool` vs `int`), avoid trusting client-supplied metadata, and serialize stable `.value` fields.
- **State:** enum members are singletons representing shared definitions; per-request state belongs in separate instances.

**Deliberate subtlety:** `DONE` and `FAILED` share a semantic rank yet remain different enum members. The capstone uses the name as a deterministic tie-breaker to make `@total_ordering` consistent with enum identity. Check how results differ when comparing the `.rank` fields instead.

In [48]:
# Final self-test: prior tests must have executed in order, and no assertion may have failed.
from collections import Counter
counts = Counter(name.split()[0] for name in CHECKED)
expected = [f"P{i}" for i in range(1, 23)]
assert all(counts[tag] > 0 for tag in expected), "A problem's checks did not run."
assert len(CHECKED) >= 100, "Insufficient executed assertions."
print(f"PASS: {len(CHECKED)} assertions passed across {len(expected)} advanced problems.")
print("PASS: workbook executed top-to-bottom without external dependencies.")

PASS: 134 assertions passed across 22 advanced problems.
PASS: workbook executed top-to-bottom without external dependencies.
